# Case Study: Bayesian logistic regression with a Laplace prior via SOUL with Metropolis-Hastings with Standardised Scaling step.

In [ ]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))  
sys.path.append(project_root)

import torch
import numpy as np
import matplotlib.pyplot as plt
import pickle
from algorithms import sig, log_p_laplace, so_mh_ss_decay, so_mh_ss
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

os.chdir(project_root)

# Obtain synthetic dataset for Laplace prior case

In [ ]:
from scipy.stats import laplace, bernoulli
# Data and design matrix
np.random.seed(3)
design_matrix = np.random.uniform(low = -1.0, high = 1.0, size = (900, 50))
x_unknown = laplace.rvs(loc = -4, size = (50, 1))
parameter_bernoulli = sig(np.matmul(design_matrix, x_unknown))
data_experiment = bernoulli.rvs(np.array(parameter_bernoulli[:, 0]), size = 900)
labels = np.expand_dims(data_experiment, axis=1)
theta_true = -4

## SOMH with Standardised Scaling

In [ ]:
# single run test

# --- SOUL MHSS Setup ---
D = 50
T = 800  # Outer optimization steps
M = 30  # MH steps per outer loop
B = 5  # Burn-in steps
delta_step = 0.08
proposal_std = 0.008 # 0.08 for soul_mh
b_scale = 1.0 # Scale parameter for the Laplace prior

# Define reasonable bounds for latent variables X
lower_bounds = np.full((D, 1), -10.0)
upper_bounds = np.full((D, 1), 5.0)

# Initializations
th0 = np.array([[-9.0]])
x0_M = np.zeros((D, 1))

# Run the adapted SOMH SS algorithm
th_list, x_samples = so_mh_ss_decay(
    log_p=log_p_laplace,
    th0=th0,
    x0_M=x0_M,
    y_l=design_matrix,
    y_f=labels,
    T=T,
    M=M,
    B=B,
    delta_step=delta_step,
    proposal_std=proposal_std,
    lower_bounds=lower_bounds,
    upper_bounds=upper_bounds,
    b=b_scale,
    gamma = 0.9995
)

print("Final estimated theta:", th_list[-1][0, 0])
print("True theta (loc parameter): -4.0")

In [ ]:
from joblib import Parallel, delayed

# --- Experiment Parameters ---
T = 1500  # Outer steps
B = 5
M = 35

# Hyper-parameters for SOMH SS
delta_step = 0.05
proposal_std = 0.008 # 0.08 for soul_mh
b_scale = 1
gamma_val = 1

# Define bounds for the standardized scaling
D = design_matrix.shape[1]
lower_bounds = np.full((D, 1), -15.0)
upper_bounds = np.full((D, 1), 10.0)

thetas = [0.0, 3.0, -13.0, -6.0, 8.0, -4.0, 5.0]

def run_single_so_mh_ss_decay(run_idx, thetas):
    #theta0_val = np.random.randint(-15, 10)
    theta0_val = thetas[run_idx]
    th0 = np.array([[float(theta0_val)]])

    # Draw X0 within bounds
    X0_M = np.random.uniform(
        low=lower_bounds[0, 0], high=upper_bounds[0, 0], size=(D, M)
    )

    th_list, x_values = so_mh_ss_decay(
        log_p=log_p_laplace,
        th0=th0,
        x0_M=X0_M,
        y_l=design_matrix,
        y_f=labels,
        T=T,
        M=M,
        B=B,
        delta_step=delta_step,
        proposal_std=proposal_std,
        lower_bounds=lower_bounds,
        upper_bounds=upper_bounds,
        b= b_scale,
        gamma= gamma_val
    )

    th_trajectory = np.array([t[0, 0] for t in th_list])
    return run_idx, theta0_val, th_trajectory, x_values


# Run parallel executions
results = Parallel(n_jobs=-2)(
    delayed(run_single_so_mh_ss_decay)(i) for i in range(len(thetas))
)

# Plotting convergence
fig = plt.figure(figsize=(10, 6))

theta_soul_ss = []
X_soul_ss = []

for run_idx, theta0_val, th_trajectory, x_values in results:
    theta_soul_ss.append(th_trajectory)
    X_soul_ss.append(x_values)

    plt.plot(
        th_trajectory,
        label=f"Run {run_idx + 1} (Init $\\theta_0={theta0_val}$)",
    )

plt.axhline(
    y=np.mean(x_unknown),
    color="black",
    linestyle="dashed",
    label=" $\\theta_{gen}$",
)

plt.title("SOMH SS Evolution Across Multiple Initializations")
plt.xlabel("Iteration (T)")
plt.ylabel("Theta Estimate ($\\theta$)")
plt.legend(loc="upper right")
plt.grid(True, alpha=0.3)

plt.show()
fig.savefig("soul_mh_ss_convergence_parallel.pdf", format="pdf")

Relative error:

In [ ]:
# Convert list of trajectories into a 2D numpy array of shape (num_runs, T + 1)
theta_soul_ss_arr = np.array(theta_soul_ss)

# Compute relative error trajectory for each run
relative_error_trajectories_ss = np.abs(theta_soul_ss_arr - theta_true) / np.abs(
    theta_true
)

# Mean relative error across all runs per iteration step
mean_relative_error_per_step_ss = np.mean(relative_error_trajectories_ss, axis=0)

# Plot Relative Error over iterations
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
for idx, err_traj in enumerate(relative_error_trajectories_ss):
    plt.plot(err_traj, alpha=0.4, label=f"Run {idx + 1}")

plt.plot(
    mean_relative_error_per_step_ss,
    color="black",
    linewidth=2,
    label="Mean Relative Distance",
)
plt.yscale("log")  # Log scale highlights convergence speed clearly
plt.title("SOUL-MHSS Relative Distance Evolution")
plt.xlabel("Iteration (T)")
plt.ylabel("Relative Distance to Generating Value (Log Scale)")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.show()

# Errors with the same $\theta_0$

In [ ]:
from joblib import Parallel, delayed

# --- Experiment Parameters ---
T = 1300  # Outer steps
B = 5
M = 35

# Hyper-parameters for SOMH SS
delta_step = 0.05
proposal_std = 0.008 # 0.08 for soul_mh
b_scale = 1
gamma_val = 1

# Define bounds for the standardized scaling
D = design_matrix.shape[1]
lower_bounds = np.full((D, 1), -15.0)
upper_bounds = np.full((D, 1), 10.0)

thetatest = 3.0

def run_single_somhss_fix_decay(run_idx, theta0_val):
    th0 = np.array([[float(theta0_val)]])
    D = design_matrix.shape[1]
    X0_M = np.random.normal(loc=theta0_val, scale=1.0, size=(D, M))

    th_list, x_values = so_mh_ss_decay(
        log_p=log_p_laplace,
        th0=th0,
        x0_M=X0_M,
        y_l=design_matrix,
        y_f=labels,
        T=T,
        M=M,
        B=B,
        delta_step=delta_step,
        proposal_std=proposal_std,
        lower_bounds=lower_bounds,
        upper_bounds=upper_bounds,
        b= b_scale,
        gamma= gamma_val
    )
    th_trajectory = np.array([t[0, 0] for t in th_list])
    return run_idx, theta0_val, th_trajectory, x_values

# Run all 5 initializations in parallel across available CPU cores (leave at least one free so the computer stays responsive)
results = Parallel(n_jobs=-2)(
    delayed(run_single_somhss_fix_decay)(i, thetatest) for i in range(5)
)

# Plot all 5 runs on one figure
fig = plt.figure(figsize=(10, 6))

theta_mh_same = []
X_mh_same = []

# Loop over the parallel results 
for run_idx, theta0_val, th_trajectory, x_values in results:
    theta_mh_same.append(th_trajectory)
    X_mh_same.append(x_values)

# Convert list of trajectories into a 2D numpy array of shape (num_runs, T + 1)
theta_mh_arr_same = np.array(theta_mh_same)

# Compute relative error trajectory for each run
relative_error_trajectories_same = np.abs(theta_mh_arr_same - theta_true) / np.abs(
    theta_true
)

# Mean relative error across all runs per iteration step
mean_relative_error_per_step_same = np.mean(relative_error_trajectories_same, axis=0)

plt.figure(figsize=(10, 5))
for idx, err_traj in enumerate(relative_error_trajectories_same):
    plt.plot(err_traj, alpha=0.4, label=f"Run {idx + 1}")

plt.plot(
    mean_relative_error_per_step_same,
    color="black",
    linewidth=2,
    label="Mean Relative Error",
)
plt.yscale("log")  # Log scale highlights convergence speed clearly
plt.title("SOMHSS Relative Error Convergence")
plt.xlabel("Iteration (T)")
plt.ylabel("Relative Error (Log Scale)")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.show()

# Save and recover the data 

In [ ]:
# Save trajectories, ground truth, and precalculated errors
np.savez_compressed(
    "so_mh_ss_results.npz",
    T=T,
    B=B,
    M=M,
    delta_step=delta_step,
    theta_true=theta_true,
    theta_soul_ss_arr=theta_soul_ss_arr,
    gamma_val = gamma_val,
    proposal_std= proposal_std,
    lower_bounds=lower_bounds,
    upper_bounds=upper_bounds,
    b_scale = b_scale,
    relative_error_trajectories_ss=relative_error_trajectories_ss,
    mean_relative_error_per_step_ss=mean_relative_error_per_step_ss,
    theta_mh_same=theta_mh_same,
    X_mh_same=X_mh_same,
    relative_error_trajectories_same=relative_error_trajectories_same,
    mean_relative_error_per_step_same=mean_relative_error_per_step_same,
)

print("Data saved successfully to 'so_mh_ss_results.npz'")

In [ ]:
# Load compressed data
data = np.load("so_mh_ss_results.npz")

T = data["T"]
theta_true = data["theta_true"]
theta_mhss_arr = data["theta_mhss"]
relative_error_trajectories = data["relative_error_trajectories"]
mean_relative_error_per_step = data["mean_relative_error_per_step"]

# Plot 1: Theta Convergence Trajectories ---
plt.figure(figsize=(10, 6))

for idx, th_trajectory in enumerate(theta_mhss_arr):
    plt.plot(th_trajectory, alpha=0.7, label=f"Run {idx + 1}")

plt.axhline(
    y=theta_true,
    color="black",
    linestyle="--",
    linewidth=1.5,
    label=r"$\theta_{gen}$",
)

plt.title("SOMH SS Evolution Across Multiple Initializations", fontsize=14, pad=12)
plt.xlabel("Iteration ($T$)", fontsize=12)
plt.ylabel(r"Theta Estimate ($\theta$)", fontsize=12)
plt.legend(loc="upper right", frameon=True, fontsize=10)
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

# Plot 2: Relative Error Convergence (Log Scale) ---
plt.figure(figsize=(10, 5))

for idx, err_traj in enumerate(relative_error_trajectories):
    plt.plot(err_traj, alpha=0.3, label=f"Run {idx + 1}")

plt.plot(
    mean_relative_error_per_step,
    color="black",
    linewidth=2,
    label="Mean Relative Distance",
)

plt.yscale("log")
plt.title("SOMH SS Relative Distance Evolution", fontsize=14, pad=12)
plt.xlabel("Iteration ($T$)", fontsize=12)
plt.ylabel("Relative Distance (Log Scale)", fontsize=12)
plt.legend(loc="upper right", frameon=True, fontsize=10)
plt.grid(True, which="both", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()